# Program 3: Rolling IV vs RV Edge Scorer

**Quant Mastery -- MBA to Market Maker**

Builds a complete volatility edge scoring system:
- Pulls 1 year of price history for any ticker
- Calculates Realized Volatility (30/60/90 day rolling)
- Pulls current options chain for Implied Volatility
- Computes IV Rank, IV Percentile, VRP signal
- Outputs SELL VOL / HOLD / BUY VOL signal with edge score
- Multi-ticker quick scan across major names
- 5-panel dark dashboard

**Curriculum Link:** Days 91-100 (Volatility Trading Systems), Days 31-40 (Statistics & Inference)

---

## Setup

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install yfinance matplotlib pandas numpy -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('All imports loaded successfully.')

## Section 1: Configuration & Price History

**What this does:** Sets the ticker and edge thresholds, then pulls 1 year of daily price history.

**Quant concept (Day 31 -- Descriptive Statistics):** All vol calculations start with log returns. We use log returns instead of simple returns because they are additive over time and approximately normally distributed.

**Key formula:**
$$r_t = \ln\left(\frac{S_t}{S_{t-1}}\right)$$

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION -- Change these to scan any ticker
# ═══════════════════════════════════════════════════════════════
TICKER = 'SPY'           # Any optionable stock/ETF
LOOKBACK_DAYS = 365      # 1 year of history for IV rank calc
RV_WINDOWS = [30, 60, 90]  # Rolling RV windows (trading days)

# Edge thresholds (in vol points)
STRONG_SELL_THRESHOLD = 5.0   # VRP > 5 = strong sell vol signal
SELL_THRESHOLD = 2.0          # VRP > 2 = sell vol
BUY_THRESHOLD = -2.0          # VRP < -2 = buy vol (rare)

print(f"{'='*60}")
print(f"  PROGRAM 3: IV vs RV Edge Scorer")
print(f"  Ticker: {TICKER}")
print(f"  Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"{'='*60}\n")

# STEP 1: Pull Historical Price Data
print("Pulling price history...")
ticker = yf.Ticker(TICKER)
end_date = datetime.now()
start_date = end_date - timedelta(days=LOOKBACK_DAYS + 100)
hist = ticker.history(start=start_date, end=end_date)

if hist.empty:
    raise ValueError(f"No data returned for {TICKER}. Check the ticker symbol.")

hist['log_return'] = np.log(hist['Close'] / hist['Close'].shift(1))
hist = hist.dropna()
print(f"  {len(hist)} trading days loaded ({hist.index[0].strftime('%Y-%m-%d')} to {hist.index[-1].strftime('%Y-%m-%d')})")

## Section 2: Rolling Realized Volatility

**What this does:** Calculates RV at 30, 60, and 90 day rolling windows.

**Quant concept (Day 93 -- RV Estimation):** Different RV windows capture different regimes:
- **30d RV:** Most responsive to recent moves. Best for short-term vol trades.
- **60d RV:** Smooths out weekly noise. Good baseline.
- **90d RV:** Captures the structural trend. Best for longer-dated trades.

**Key formula:**
$$\text{RV}_n = \text{std}(r_{t-n+1}, \ldots, r_t) \times \sqrt{252}$$

In [ ]:
print("Calculating Realized Volatility...")
for window in RV_WINDOWS:
    col_name = f'RV_{window}d'
    hist[col_name] = hist['log_return'].rolling(window=window).std() * np.sqrt(252) * 100
    current_rv = hist[col_name].iloc[-1]
    print(f"  RV {window}-day: {current_rv:.1f}%")

## Section 3: Current Implied Volatility from Options

**What this does:** Pulls the options chain for ~30 DTE and extracts ATM IV. Also computes vol skew (25-delta put vs ATM).

**Quant concept (Day 94 -- IV Surface Extraction):** ATM IV is the cleanest measure of the market's vol expectation. We average call and put ATM IV to mitigate bid-ask noise. The skew (OTM put IV - ATM IV) measures tail risk pricing.

In [ ]:
print("\nPulling options chain for IV...")
try:
    expirations = ticker.options
    if not expirations:
        raise ValueError("No options data available")

    today = datetime.now().date()
    target_dte = 30
    best_exp = None
    best_diff = float('inf')

    for exp_str in expirations:
        exp_date = datetime.strptime(exp_str, '%Y-%m-%d').date()
        dte = (exp_date - today).days
        if 14 <= dte <= 60:
            diff = abs(dte - target_dte)
            if diff < best_diff:
                best_diff = diff
                best_exp = exp_str

    if best_exp is None:
        best_exp = expirations[min(2, len(expirations)-1)]

    exp_date = datetime.strptime(best_exp, '%Y-%m-%d').date()
    dte = (exp_date - today).days
    print(f"  Using expiration: {best_exp} ({dte} DTE)")

    chain = ticker.option_chain(best_exp)
    calls = chain.calls
    puts = chain.puts
    current_price = hist['Close'].iloc[-1]
    print(f"  Current price: ${current_price:.2f}")

    calls['dist'] = abs(calls['strike'] - current_price)
    puts['dist'] = abs(puts['strike'] - current_price)
    atm_call = calls.loc[calls['dist'].idxmin()]
    atm_put = puts.loc[puts['dist'].idxmin()]

    atm_iv = (atm_call['impliedVolatility'] + atm_put['impliedVolatility']) / 2 * 100
    print(f"  ATM Call IV: {atm_call['impliedVolatility']*100:.1f}%")
    print(f"  ATM Put IV:  {atm_put['impliedVolatility']*100:.1f}%")
    print(f"  ATM IV (avg): {atm_iv:.1f}%")

    otm_put_strike = current_price * 0.95
    puts['dist_otm'] = abs(puts['strike'] - otm_put_strike)
    otm_put = puts.loc[puts['dist_otm'].idxmin()]
    skew = otm_put['impliedVolatility'] * 100 - atm_iv
    print(f"  Vol skew (25d put - ATM): {skew:+.1f} pts")

except Exception as e:
    print(f"  Options data issue: {e}")
    print(f"  Using VIX as IV proxy...")
    vix = yf.Ticker('^VIX')
    vix_hist = vix.history(period='5d')
    atm_iv = vix_hist['Close'].iloc[-1]
    skew = 0
    dte = 30
    print(f"  VIX (IV proxy): {atm_iv:.1f}%")

## Section 4: IV Rank & IV Percentile

**What this does:** Calculates where current IV sits relative to its 1-year range.

**Quant concept (Day 95 -- IV Rank vs IV Percentile):**

- **IV Rank** = where current IV sits in the 1-year high/low range:
$$\text{IVR} = \frac{\text{IV}_{\text{current}} - \text{IV}_{\text{1yr low}}}{\text{IV}_{\text{1yr high}} - \text{IV}_{\text{1yr low}}} \times 100$$

- **IV Percentile** = what percentage of days in the past year had IV *lower* than today.

IVR > 50 = elevated IV = good time to sell premium. IVR < 25 = cheap IV = bad time to sell.

In [ ]:
print("\nComputing IV metrics...")

try:
    vix = yf.Ticker('^VIX')
    vix_hist = vix.history(start=start_date, end=end_date)
    iv_series = vix_hist['Close']
    iv_source = 'VIX'
except:
    iv_series = hist['RV_30d'].dropna()
    iv_source = 'RV_30d proxy'

iv_1yr = iv_series.tail(252)
iv_high = iv_1yr.max()
iv_low = iv_1yr.min()
iv_current = atm_iv

iv_rank = (iv_current - iv_low) / (iv_high - iv_low) * 100 if iv_high != iv_low else 50
iv_percentile = (iv_1yr < iv_current).sum() / len(iv_1yr) * 100

print(f"  IV source: {iv_source}")
print(f"  1-year IV range: {iv_low:.1f}% to {iv_high:.1f}%")
print(f"  IV Rank: {iv_rank:.0f}th (current vs 1yr range)")
print(f"  IV Percentile: {iv_percentile:.0f}th (% of days below current)")

## Section 5: VRP & Edge Score

**What this does:** Computes the Vol Risk Premium across multiple windows and generates a composite edge score.

**Quant concept (Day 96 -- Edge Scoring):** The composite VRP weights shorter windows more heavily (50% RV30, 30% RV60, 20% RV90). The edge score adjusts VRP by IV rank -- selling vol when IV rank is high AND VRP is positive gives the strongest signal.

$$\text{VRP}_{\text{composite}} = 0.5 \times \text{VRP}_{30} + 0.3 \times \text{VRP}_{60} + 0.2 \times \text{VRP}_{90}$$
$$\text{Edge Score} = \text{VRP}_{\text{comp}} \times \left(1 + \frac{\text{IVR} - 50}{100}\right)$$

In [ ]:
print("Computing Vol Risk Premium & Edge Score...")
rv_30 = hist['RV_30d'].iloc[-1]
rv_60 = hist['RV_60d'].iloc[-1]
rv_90 = hist['RV_90d'].iloc[-1]

vrp_30 = atm_iv - rv_30
vrp_60 = atm_iv - rv_60
vrp_90 = atm_iv - rv_90

print(f"  VRP (IV - RV30): {vrp_30:+.1f} vol pts")
print(f"  VRP (IV - RV60): {vrp_60:+.1f} vol pts")
print(f"  VRP (IV - RV90): {vrp_90:+.1f} vol pts")

vrp_composite = 0.5 * vrp_30 + 0.3 * vrp_60 + 0.2 * vrp_90
edge_score = vrp_composite * (1 + (iv_rank - 50) / 100)

print(f"\n  Composite VRP: {vrp_composite:+.1f}")
print(f"  Edge Score: {edge_score:+.1f}")

# Generate signal
if vrp_composite > STRONG_SELL_THRESHOLD and iv_rank > 50:
    signal = "STRONG SELL VOL"
    signal_detail = "IV is expensive AND overpricing realized moves. Premium selling is highly favorable."
    signal_color = '#00ff88'
elif vrp_composite > SELL_THRESHOLD:
    signal = "SELL VOL"
    signal_detail = "Positive VRP -- IV is overpricing. Standard vol-selling conditions."
    signal_color = '#88ff88'
elif vrp_composite < BUY_THRESHOLD:
    signal = "BUY VOL"
    signal_detail = "Rare: IV is UNDER-pricing realized moves. Consider long vol or sit out."
    signal_color = '#ff4444'
else:
    signal = "HOLD / NO EDGE"
    signal_detail = "VRP is too narrow. No clear edge -- wait for better setup."
    signal_color = '#ffaa00'

print(f"\n{'='*60}")
print(f"  SIGNAL: {signal}")
print(f"  {signal_detail}")
print(f"{'='*60}")

## Section 6: 5-Panel Dashboard

**What this does:** Renders the full edge scanner dashboard:
1. Price chart (6M)
2. Rolling RV vs current IV
3. VRP bar chart over time (the edge visualization)
4. IV Rank gauge
5. Signal summary card

In [ ]:
print("Rendering dashboard...")

plt.style.use('dark_background')
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0a0a0a')
gs = gridspec.GridSpec(3, 2, hspace=0.35, wspace=0.25,
                       left=0.06, right=0.97, top=0.92, bottom=0.05)

fig.suptitle(f'{TICKER} -- IV vs RV Edge Scanner  |  {datetime.now().strftime("%Y-%m-%d")}',
             fontsize=16, fontweight='bold', color='white', y=0.97)

# Panel 1: Price chart
ax1 = fig.add_subplot(gs[0, 0])
price_data = hist['Close'].tail(180)
ax1.plot(price_data.index, price_data.values, color='white', linewidth=1.2, alpha=0.9)
ax1.fill_between(price_data.index, price_data.values, alpha=0.05, color='white')
ax1.set_title(f'{TICKER} Price (6M)', fontsize=11, fontweight='bold', color='white')
ax1.set_ylabel('Price ($)', fontsize=9, color='#888')
ax1.tick_params(colors='#888', labelsize=8)
ax1.grid(alpha=0.1)
for spine in ax1.spines.values(): spine.set_color('#333')

# Panel 2: Rolling RV comparison
ax2 = fig.add_subplot(gs[0, 1])
rv_data = hist[['RV_30d', 'RV_60d', 'RV_90d']].tail(180)
ax2.plot(rv_data.index, rv_data['RV_30d'], color='#ffffff', linewidth=1.5, label='RV 30d', alpha=0.9)
ax2.plot(rv_data.index, rv_data['RV_60d'], color='#aaaaaa', linewidth=1.2, label='RV 60d', alpha=0.7)
ax2.plot(rv_data.index, rv_data['RV_90d'], color='#666666', linewidth=1.0, label='RV 90d', alpha=0.6)
ax2.axhline(y=atm_iv, color='#00ff88', linestyle='--', linewidth=1.5, label=f'Current IV ({atm_iv:.1f}%)', alpha=0.8)
ax2.set_title('Realized Vol vs Implied Vol', fontsize=11, fontweight='bold', color='white')
ax2.set_ylabel('Volatility (%)', fontsize=9, color='#888')
ax2.legend(fontsize=7, loc='upper right', framealpha=0.3)
ax2.tick_params(colors='#888', labelsize=8)
ax2.grid(alpha=0.1)
for spine in ax2.spines.values(): spine.set_color('#333')

# Panel 3: VRP over time
ax3 = fig.add_subplot(gs[1, :])
if iv_source == 'VIX':
    aligned_iv = iv_series.reindex(hist.index, method='ffill')
    hist['iv_proxy'] = aligned_iv
else:
    hist['iv_proxy'] = hist['RV_30d'].shift(-20)
vrp_series = hist['iv_proxy'] - hist['RV_30d']
vrp_plot = vrp_series.tail(180).dropna()
colors_vrp = ['#00ff88' if v > SELL_THRESHOLD else '#ff4444' if v < BUY_THRESHOLD else '#ffaa00'
              for v in vrp_plot.values]
ax3.bar(vrp_plot.index, vrp_plot.values, color=colors_vrp, alpha=0.7, width=1.5)
ax3.axhline(y=0, color='white', linewidth=0.8, alpha=0.3)
ax3.axhline(y=SELL_THRESHOLD, color='#00ff88', linewidth=0.8, linestyle='--', alpha=0.4)
ax3.axhline(y=BUY_THRESHOLD, color='#ff4444', linewidth=0.8, linestyle='--', alpha=0.4)
ax3.set_title('Vol Risk Premium (IV - RV30)  |  Green = Sell Vol Edge  |  Red = Buy Vol',
              fontsize=11, fontweight='bold', color='white')
ax3.set_ylabel('VRP (vol pts)', fontsize=9, color='#888')
ax3.tick_params(colors='#888', labelsize=8)
ax3.grid(alpha=0.1)
for spine in ax3.spines.values(): spine.set_color('#333')

# Panel 4: IV Rank gauge
ax4 = fig.add_subplot(gs[2, 0])
gauge_colors = ['#ff4444' if iv_rank < 25 else '#ffaa00' if iv_rank < 50 else '#00ff88' if iv_rank < 75 else '#00ffff']
ax4.barh([0], [iv_rank], height=0.5, color=gauge_colors[0], alpha=0.8)
ax4.barh([0], [100], height=0.5, color='#222', alpha=0.3)
ax4.set_xlim(0, 100)
ax4.set_yticks([])
ax4.set_title(f'IV Rank: {iv_rank:.0f}th  |  IV Percentile: {iv_percentile:.0f}th',
              fontsize=11, fontweight='bold', color='white')
ax4.axvline(x=25, color='#444', linewidth=0.5)
ax4.axvline(x=50, color='#444', linewidth=0.5)
ax4.axvline(x=75, color='#444', linewidth=0.5)
ax4.text(12.5, -0.7, 'LOW', ha='center', fontsize=8, color='#666')
ax4.text(37.5, -0.7, 'MID', ha='center', fontsize=8, color='#666')
ax4.text(62.5, -0.7, 'HIGH', ha='center', fontsize=8, color='#666')
ax4.text(87.5, -0.7, 'EXTREME', ha='center', fontsize=8, color='#666')
ax4.set_ylim(-1.2, 1)
ax4.tick_params(colors='#888', labelsize=8)
for spine in ax4.spines.values(): spine.set_color('#333')

# Panel 5: Signal summary card
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_xlim(0, 10)
ax5.set_ylim(0, 10)
ax5.set_xticks([])
ax5.set_yticks([])
for spine in ax5.spines.values(): spine.set_color('#333')
ax5.text(5, 8.5, signal, fontsize=16, fontweight='bold', color=signal_color, ha='center', va='center')
ax5.text(5, 7.0, signal_detail, fontsize=8, color='#888', ha='center', va='center', wrap=True,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1a1a', edgecolor='#333'))
metrics = [
    f"ATM IV: {atm_iv:.1f}%",
    f"RV 30d: {rv_30:.1f}%  |  RV 60d: {rv_60:.1f}%",
    f"VRP: {vrp_composite:+.1f} pts  |  Edge: {edge_score:+.1f}",
    f"IV Rank: {iv_rank:.0f}  |  Skew: {skew:+.1f}",
    f"Expiry: {best_exp} ({dte} DTE)",
]
for i, m in enumerate(metrics):
    ax5.text(5, 5.0 - i * 1.0, m, fontsize=9, color='#ccc', ha='center', va='center', family='monospace')
ax5.set_title('EDGE SCANNER SIGNAL', fontsize=11, fontweight='bold', color='white')

plt.show()

## Section 7: Multi-Ticker Quick Scan

**What this does:** Scans 8 major tickers for their current VRP and generates a quick signal for each.

**Quant concept (Day 97 -- Universe Screening):** The best vol trades come from scanning a universe and finding the fattest VRP, not from trading one ticker blindly. This is like game selection in poker -- find the table with the biggest edge.

In [ ]:
print(f"{'='*60}")
print(f"  MULTI-TICKER QUICK SCAN")
print(f"{'='*60}")

scan_tickers = ['SPY', 'QQQ', 'IWM', 'AAPL', 'MSFT', 'TSLA', 'AMZN', 'NVDA']
print(f"\n  {'Ticker':<8} {'RV 30d':>8} {'ATM IV':>8} {'VRP':>8} {'Signal':<20}")
print(f"  {'_'*56}")

for t in scan_tickers:
    try:
        tk = yf.Ticker(t)
        h = tk.history(period='6mo')
        if h.empty: continue
        h['lr'] = np.log(h['Close'] / h['Close'].shift(1))
        rv = h['lr'].rolling(30).std().iloc[-1] * np.sqrt(252) * 100

        exps = tk.options
        if exps:
            exp = None
            for e in exps:
                d = (datetime.strptime(e, '%Y-%m-%d').date() - today).days
                if 14 <= d <= 60:
                    exp = e
                    break
            if exp is None: exp = exps[min(1, len(exps)-1)]
            ch = tk.option_chain(exp)
            cp = h['Close'].iloc[-1]
            ch.calls['dist'] = abs(ch.calls['strike'] - cp)
            atm = ch.calls.loc[ch.calls['dist'].idxmin()]
            iv = atm['impliedVolatility'] * 100
        else:
            iv = rv * 1.15

        vrp = iv - rv
        if vrp > STRONG_SELL_THRESHOLD:
            sig = "STRONG SELL"
        elif vrp > SELL_THRESHOLD:
            sig = "SELL VOL"
        elif vrp < BUY_THRESHOLD:
            sig = "BUY VOL"
        else:
            sig = "HOLD"

        print(f"  {t:<8} {rv:>7.1f}% {iv:>7.1f}% {vrp:>+7.1f}  {sig}")
    except Exception as e:
        print(f"  {t:<8} {'error':>8} -- {str(e)[:30]}")

print(f"\n  Done. Run daily before market open for best results.")

---
## Exercise 1: Scan a Different Universe

Replace `scan_tickers` with sector-specific names:
```python
scan_tickers = ['XLF', 'XLE', 'XLK', 'XLV', 'XLI', 'XLP', 'XLU', 'XLB']
```
Which sector has the highest VRP right now? Which sector's vol is cheapest?

## Exercise 2: Modify the DTE Target

Change `target_dte = 30` to `target_dte = 7` (weekly options). How does ATM IV differ for weeklies vs monthlies? Why is weekly IV typically higher?

## Exercise 3: Adjust Edge Thresholds

Try tighter thresholds:
```python
STRONG_SELL_THRESHOLD = 3.0
SELL_THRESHOLD = 1.0
BUY_THRESHOLD = -1.0
```
More tickers now show as SELL signals. Is that good or bad? What is the tradeoff between sensitivity and false positives?

---
## Key Takeaways

1. **IV Rank tells you WHEN to sell vol.** High IVR (>50) means IV is elevated relative to its range -- premium is rich.

2. **VRP tells you IF there is an edge.** Positive VRP = IV overpricing realized moves = structural edge for sellers.

3. **Composite scoring reduces noise.** Weighting multiple RV windows and combining with IV rank creates a more robust signal than any single metric.

4. **Universe scanning is game selection.** The best traders don't force trades -- they scan and wait for the fattest setup.

5. **This runs daily.** The edge changes every day. Build the habit of running this before market open.

**Next:** Program 4 applies this framework specifically to earnings events, where IV crush is most dramatic.